In [1]:
import kagglehub
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import os
from sklearn.model_selection import train_test_split

# Download latest version
path = kagglehub.dataset_download("alexteboul/diabetes-health-indicators-dataset")

print("Path to dataset files:", path)

file_path = os.path.join(path, "diabetes_012_health_indicators_BRFSS2015.csv")
df = pd.read_csv(file_path)

#converting 0 and 1s to 0 and 2s to 1
df["Diabetes_012"] = df["Diabetes_012"].replace({1:0,2:1})
print("New distribution according to 0,1 only:\n", df["Diabetes_012"].value_counts(normalize = True))
df = df.rename(columns={"Diabetes_012": "Diabetes_binary"})

#removing duplicate entries
print("Total number of duplicates:\n", df.duplicated().sum())
df = df.drop_duplicates()
print(f"After removing duplicates:\n {df.shape}")

#X and y variables
X = df.drop("Diabetes_binary", axis = 1)
y = df["Diabetes_binary"]

#training testing split
X_train, X_validtest, y_train, y_validtest = train_test_split(X, y, test_size = 0.3, random_state = 42, stratify = y)

#splitting into validation and testing
X_val, X_test, y_val, y_test = train_test_split(X_validtest, y_validtest, test_size=0.5, random_state=42, stratify=y_validtest)

print(f"Training data size: {X_train.shape[0]} & Validation data size: {X_val.shape[0]} & Testing data size: {X_test.shape[0]}")


Path to dataset files: C:\Users\Hp\.cache\kagglehub\datasets\alexteboul\diabetes-health-indicators-dataset\versions\1
New distribution according to 0,1 only:
 Diabetes_012
0.0    0.860667
1.0    0.139333
Name: proportion, dtype: float64
Total number of duplicates:
 24206
After removing duplicates:
 (229474, 22)
Training data size: 160631 & Validation data size: 34421 & Testing data size: 34422


In [2]:
import numpy as np

# Helper functions for the math
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def sigmoid_derivative(z):
    s = sigmoid(z)
    return s * (1 - s)

# 1. PROGRAM FORWARD PROPAGATION
def forward_propagation(X, W1, b1, W2, b2):
    """Passes the data forward through the network to make a guess."""
    # Layer 1 (Hidden)
    Z1 = np.dot(W1, X) + b1
    A1 = sigmoid(Z1)
    
    # Layer 2 (Output)
    Z2 = np.dot(W2, A1) + b2
    A2 = sigmoid(Z2)
    
    return Z1, A1, Z2, A2

# 2. PROGRAM COST FUNCTION
def compute_cost(A2, Y):
    """Calculates how wrong the model's guess was (Log-Loss)."""
    m = Y.shape[1] # Number of examples
    cost = -(1/m) * np.sum(Y * np.log(A2) + (1 - Y) * np.log(1 - A2))
    return cost

# 3. PROGRAM BACKWARD PROPAGATION
def backward_propagation(X, Y, Z1, A1, A2, W2):
    """Calculates the gradients (adjustments) to fix the model's errors."""
    m = X.shape[1]
    
    # Layer 2 adjustments
    dZ2 = A2 - Y
    dW2 = (1/m) * np.dot(dZ2, A1.T)
    db2 = (1/m) * np.sum(dZ2, axis=1, keepdims=True)
    
    # Layer 1 adjustments
    dA1 = np.dot(W2.T, dZ2)
    dZ1 = dA1 * sigmoid_derivative(Z1)
    dW1 = (1/m) * np.dot(dZ1, X.T)
    db1 = (1/m) * np.sum(dZ1, axis=1, keepdims=True)
    
    return dW1, db1, dW2, db2

print("Functions for Forward Prop, Cost, and Backprop successfully defined!")

Functions for Forward Prop, Cost, and Backprop successfully defined!


In [3]:
import numpy as np

# Fonction sigmoïde
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

# Fonction coût (log-loss)
def compute_cost(A, y):
    m = y.shape[0]
    # Small epsilon added to avoid log(0) errors
    cost = -(1/m) * np.sum(y * np.log(A + 1e-15) + (1 - y) * np.log(1 - A + 1e-15))
    return cost

# Régression logistique avec descente de gradient
def logistic_regression(X, y, alpha=0.01, n_iter=1000):
    m, n = X.shape
    
    # Ajouter une colonne de 1 à X pour le biais (bias)
    X = np.hstack((X, np.ones((m, 1))))
    
    # Initialiser les poids (weights)
    W = np.zeros((n + 1, 1))
    y = y.reshape(-1, 1)
    
    for i in range(n_iter):
        # Étape 1 : prédiction (Forward Prop)
        Z = np.dot(X, W)
        A = sigmoid(Z)
        
        # Étape 2 : coût
        cost = compute_cost(A, y)
        
        # Étape 3 : gradient
        grad = (1/m) * np.dot(X.T, (A - y))
        
        # Étape 4 : mise à jour (Gradient Descent)
        W -= alpha * grad
        
        # Optionnel : afficher le coût toutes les 100 itérations
        if i % 100 == 0:
            print(f"Iteration {i}, coût = {cost:.4f}")
            
    return W

# Exemple d'utilisation
if __name__ == "__main__":
    # Données factices (X : 2 features, y : labels 0/1)
    X = np.array([
        [0.5, 1.5],
        [1.0, 1.8],
        [2.0, 2.5],
        [3.0, 3.5],
        [4.0, 4.5]
    ])
    
    y = np.array([0, 0, 0, 1, 1]) # Labels
    
    # Entraînement du modèle
    W = logistic_regression(X, y, alpha=0.1, n_iter=1000)
    
    print("\nParamètres finaux (W) :")
    print(W)

Iteration 0, coût = 0.6931
Iteration 100, coût = 0.3823
Iteration 200, coût = 0.2808
Iteration 300, coût = 0.2306
Iteration 400, coût = 0.2000
Iteration 500, coût = 0.1790
Iteration 600, coût = 0.1633
Iteration 700, coût = 0.1510
Iteration 800, coût = 0.1411
Iteration 900, coût = 0.1327

Paramètres finaux (W) :
[[ 3.17421928]
 [-0.87676546]
 [-5.03648975]]


In [4]:
import numpy as np

def sigmoid(z): return 1/(1+np.exp(-z))

# données
x = np.array([[0.7, 0.9]])
y = np.array([[1.0]])

W1 = np.array([[0.3, -0.2],[0.6, 0.1]])
b1 = np.array([[0.0, 0.1]])
W2 = np.array([[0.4],[-0.3]])
b2 = np.array([[0.05]])
alpha = 0.1

# forward
Z1 = x.dot(W1) + b1
A1 = sigmoid(Z1)
Z2 = A1.dot(W2) + b2
A2 = sigmoid(Z2)
cost = - (y*np.log(A2) + (1-y)*np.log(1-A2))

# backward
dZ2 = A2 - y
dW2 = A1.T.dot(dZ2)
db2 = dZ2
dA1 = dZ2.dot(W2.T)
dZ1 = dA1 * (A1 * (1 - A1))
dW1 = x.T.dot(dZ1)
db1 = dZ1

# update
W1 -= alpha * dW1
b1 -= alpha * db1
W2 -= alpha * dW2
b2 -= alpha * db2

print("A2 =", A2)
print("cost =", cost)
print("dW1 =", dW1)
print("dW2 =", dW2)
print("W1_new =", W1)
print("W2_new =", W2)

A2 = [[0.5418822]]
cost = [[0.61270665]]
dW1 = [[-0.02795004  0.02403616]
 [-0.03593577  0.03090363]]
dW2 = [[-0.31114385]
 [-0.23478418]]
W1_new = [[ 0.302795   -0.20240362]
 [ 0.60359358  0.09690964]]
W2_new = [[ 0.43111439]
 [-0.27652158]]
